<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Practical: Data Quality with PySpark (Titanic)

*Session 5 · Notebook 02.03 · Practical · Student version*

## About this notebook

This practical applies the data cleaning toolkit (notebook 02.02) to the Titanic passenger list: read it with an explicit schema, inspect it, check duplicates and business rules, clean the text, and fill the missing values with decisions you can justify. By the end the data is clean enough to feed the classification models in the MLlib notebooks, and you will have used a cleaned column (the passenger's title) to explain who survived.

## About the exercises

Each task appears as two cells:

1. A markdown cell describing what to do (sometimes with a `> *Hint:*`).
2. A `# Your turn` cell for you to write your answer.

This is the student version, so the answers are not included. Work top to bottom: later tasks reuse variables from earlier answers.

## About the data

The **Titanic** dataset, in two files: `train_titanic.csv` (891 passengers, with the `Survived` label) and `test_titanic.csv` (418 passengers, without it). Cleaning decisions (fill values, for example) are always taken from the **training** data and then applied to both.

| Column | Type | Description |
|---|---|---|
| `PassengerId` | int | Unique id for each passenger |
| `Survived` | int (0/1) | 0 = did not survive, 1 = survived (training file only) |
| `Pclass` | int | Ticket class (1 = first, 2 = second, 3 = third) |
| `Name` | string | Passenger name, as `Surname, Title. First names` |
| `Sex` | string | male or female |
| `Age` | double | Age in years (fractions for babies) |
| `SibSp` | int | Number of siblings and spouses aboard |
| `Parch` | int | Number of parents and children aboard |
| `Ticket` | string | Ticket number |
| `Fare` | double | Ticket price |
| `Cabin` | string | Cabin number |
| `Embarked` | string | Port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton) |

Paths come from the session `config.py`; during the course the data is read from the S3 bucket.

## Index

0. [Setup](#setup)
1. [Read with a schema and inspect](#inspect)
2. [Duplicates and rules](#rules)
3. [Cleaning text: names and titles](#text)
4. [Missing values](#missing)
5. [Capstone](#capstone)
6. [Moving forward](#forward)

<a id="setup"></a>
## 0. Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))   # finds config.py next to this notebook or one level up

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

from config import pandas_bridge, session_datasets, session_datasets_http

# On Databricks this returns the session that already exists; on a laptop it starts one.
spark = SparkSession.builder.appName("05_02_03_Practical_DataQuality").getOrCreate()
print("Spark version:", spark.version)

In [ ]:
def null_counts(frame):
    """One row with the number of nulls in every column."""
    return frame.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in frame.columns])

<a id="inspect"></a>
## 1. Read with a schema and inspect

### Exercise 1: Read both files with an explicit schema

Write the schema of the **training** file as a DDL string (types as in the table above), read `train_titanic` into `df` with it, then read `test_titanic` into `df_test` with the same schema **minus** `Survived`. Print the row counts and the training schema.

> *Hint:* Column order in the schema must match the file. `Ticket` is text: some tickets contain letters.

In [ ]:
# Your turn

### Exercise 2: Inspect

Count the nulls in every column of `df` and `df_test` (use `null_counts`), and show a `summary()` of `Age` and `Fare` for the training data. Which columns need attention?

In [ ]:
# Your turn

<a id="rules"></a>
## 2. Duplicates and rules

### Exercise 3: Duplicates

Count the exact duplicate rows in `df`, and the duplicates of the business key `PassengerId`.

In [ ]:
# Your turn

### Exercise 4: Rule checks

Add a `dq_issue` column that holds the first rule each row breaks: `Survived` must be 0 or 1, `Pclass` 1, 2 or 3, `Sex` male or female, `Fare` not negative, and `Age` (when present) between 0 and 100. Count the issues. Then look separately at the passengers with a fare of exactly 0: how many are there, and what do their tickets have in common?

> *Hint:* A null age makes `between` null, which `when` treats as not true; test `isNotNull()` explicitly.

In [ ]:
# Your turn

<a id="text"></a>
## 3. Cleaning text: names and titles

### Exercise 5: Stray spaces

How many `Name` values have spaces at the start or the end? Fix them in both DataFrames with `F.trim`.

> *Hint:* Wrapping the value in brackets with `F.concat` makes trailing spaces visible.

In [ ]:
# Your turn

### Exercise 6: Extract the title

Every name follows the pattern `Surname, Title. First names`. Extract the title (the text between the comma and the first full stop) into a column `Title` in both DataFrames, and count the passengers per title in the training data.

> *Hint:* `F.regexp_extract(col, pattern, 1)` returns the text matched by the first group in brackets.

In [ ]:
# Your turn

### Exercise 7: Standardise the titles

Map the French forms to English (`Mlle` and `Ms` to `Miss`, `Mme` to `Mrs`), keep `Mr`, `Mrs`, `Miss` and `Master`, and group every other title as `Other`. Do it in both DataFrames, then show the counts and the survival rate per title in the training data.

In [ ]:
# Your turn

<a id="missing"></a>
## 4. Missing values

### Exercise 8: Flag and fill Cabin

`Cabin` is missing for about 77% of the training passengers: too sparse to fill with a value, but whether a cabin was recorded is informative. In both DataFrames, add `HasCabin` (1 if a cabin is recorded, else 0) and fill the missing cabins with `"N"`. Compare the survival rate with and without a recorded cabin.

In [ ]:
# Your turn

### Exercise 9: Fill Fare and Embarked

One test passenger has no `Fare`: fill it with the **training** median fare. Two training passengers have no `Embarked`: fill it with the most common port in the training data.

> *Hint:* Fill values come from the training data only, so the test data cannot leak into what the model learns.

In [ ]:
# Your turn

### Exercise 10: Fill Age by title

Filling every missing age with one overall median would make missing boys (`Master`) as old as adult men. Instead, compute the **median age per `Title`** in the training data and use it to fill the missing ages in both DataFrames. Confirm that no nulls remain in `df` and `df_test`.

> *Hint:* Join the medians on `Title`, then `F.coalesce(Age, title_median_age)` keeps the real age when there is one.

In [ ]:
# Your turn

<a id="capstone"></a>
## 5. Capstone

### Capstone: A survival profile

With the cleaned `df`, build one summary table showing, for each combination of `Pclass` and `Title`: the number of passengers, the survival rate, the average age, the average fare and the share with a recorded cabin, sorted by survival rate. What does the title add that `Sex` alone would not tell you?

In [ ]:
# Your turn

<a id="forward"></a>
## 6. Moving forward

The data is now clean and almost ready for a model. The MLlib notebooks (03.05 and 03.06) turn it into a survival classifier with **Spark MLlib**:

1. Choose the target (`Survived`) and the features, including the new `Title` and `HasCabin`.
2. Encode the categorical features (`Sex`, `Embarked`, `Title`) with `StringIndexer` and `OneHotEncoder`, and assemble everything into one `features` vector with `VectorAssembler`.
3. Split into training and test sets and fit a classifier such as `LogisticRegression` or `RandomForestClassifier`.
4. Evaluate the predictions with `BinaryClassificationEvaluator`.

On a laptop, release the resources when you finish with `spark.stop()`. **Do not** do this on Databricks, where the platform manages the session.

In [ ]:
# spark.stop()   # local environments only; never on Databricks

### Further Reading

- Apache Spark: [String functions (`trim`, `regexp_extract`, ...)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#string-functions)
- Apache Spark: [DataFrameNaFunctions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameNaFunctions.html)
- Apache Spark: [ML classification and regression](https://spark.apache.org/docs/latest/ml-classification-regression.html)